In [5]:
import pdfplumber
from pathlib import Path
import re

In [6]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [11]:
file = ROOT / Path("examenes_quimica/2019_quimica/2019_quimica_reserva_a.pdf")

In [8]:
def get_exercise_box(file):
    exercise_positions = {}
    with pdfplumber.open(file) as pdf:
            for index, p in enumerate(pdf.pages):
                words = p.extract_words()
                words_page_dic = {}
                for w in words:
                    words_page_dic[w["text"]] = {k:w[k] for k in ["x0","top", "bottom"]}
                exercise_positions[f"page_{index}"] = {k:v for k,v in words_page_dic.items()
                                                    if k in re.findall(r"\d{1}\.-", k)}

    exercise_positions_list = []
    for page_index in range(2):
        for k,v in exercise_positions[f"page_{page_index}"].items():
            dic = v
            dic["text"] = k
            dic["page"] = page_index
            exercise_positions_list.append(dic)

    return exercise_positions_list

In [9]:
MARGEN_LATERAL = 10    # puntos de aire a izquierda y derecha
MARGEN_SUPERIOR = 8    # puntos de aire por encima
RECORTE_INFERIOR = 0.5   # puntos que se recortan por abajo
RESOLUCION = 200  # dpi del PNG

def crop_exercises(pdf_path, start_coordinates, output_folder):
    """
    inicios: lista de dicts con 'page' (índice de página), 'top', 'bottom', 'x0'
             de la palabra con la que empieza cada ejercicio.
    """
    with pdfplumber.open(pdf_path) as pdf:
        # Ordenamos por página y luego por altura, por si vienen desordenados
        start_coordinates = sorted(start_coordinates, key=lambda w: (w["page"], w["top"]))

        for index, exercise_start in enumerate(start_coordinates):
            page = pdf.pages[exercise_start["page"]]

            # ¿El siguiente ejercicio está en la misma página?
            next_exercise = start_coordinates[index + 1] if index + 1 < len(start_coordinates) else None
            if next_exercise and next_exercise["page"] == exercise_start["page"]:
                limite_inf = next_exercise["top"]
            else:
                limite_inf = page.height

            # Zona "bruta" del ejercicio: de su inicio al siguiente
            zone = page.crop((0, exercise_start["top"], page.width, limite_inf))

            # Ajustamos al texto real para no arrastrar espacio en blanco
            # page.crop() incluye también la primera línea del ejercicio siguiente
            # (recortada a altura 0 en top == bottom == limite_inf): la descartamos
            zone_words = [w for w in zone.extract_words() if w["top"] < limite_inf]
            bottom_real = max(w["bottom"] for w in zone_words)
            x1_real = max(w["x1"] for w in zone_words)

            bbox = [
                max(exercise_start["x0"] - MARGEN_LATERAL, 0),
                max(exercise_start["top"] - MARGEN_SUPERIOR, 0),
                page.width - 2,
                # min(x1_real + MARGEN_LATERAL, page.width),
                min(bottom_real - RECORTE_INFERIOR, page.height),
            ]
            
            bottoms_above = [
                w["bottom"] for w in page.extract_words()
                if w["bottom"] <= exercise_start["top"]
            ]
            if bottoms_above:
                bbox[1] = max(bbox[1], max(bottoms_above) + 0.5)

            recorte = page.crop(bbox)
            if not isinstance(output_folder, Path):
                destination_folder = Path(output_folder)
            Path.mkdir(destination_folder, exist_ok=True)
            recorte.to_image(resolution=RESOLUCION).save(
                f"{destination_folder}/ejercicio_{index + 1}.png"
            )

Aplicamos el recorte a los ejercicios anteriores a 2020

In [10]:
def crop_exercises_from_exam(file, output_folder):
    if not isinstance(output_folder, Path):
        destination_folder = Path(output_folder)
    Path.mkdir(destination_folder, exist_ok=True)

    # get the positions of the boxes from each exercise
    box_positions = get_exercise_box(file)

    # crop the exercises
    crop_exercises(file, box_positions, output_folder)

In [12]:
crop_exercises_from_exam(file, output_folder="2019_quimica_img")